In [ ]:
import os
import json
import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12


# ========== CONFIGURATION ==========
# Set your log directory path here
LOG_DIR = 'logs/HybridMR_255opts_c450e1a2'  # Replace with your actual log directory

# Verify directory exists
if not os.path.exists(LOG_DIR):
    raise FileNotFoundError(f"Log directory not found: {LOG_DIR}")

print(f"Analyzing results from: {LOG_DIR}")


# Load config.pt
config_path = os.path.join(LOG_DIR, 'config.pt')
config_data = torch.load(config_path)

optimizer_configs = config_data['optimizer_configs']
print(f"Total optimizers: {len(optimizer_configs)}")
print(f"\nFirst few optimizer configs:")
for i, cfg in enumerate(optimizer_configs[:3]):
    print(f"  {i+1}. {cfg}")



# Extract final test accuracies and group by optimizer type
results = []

for opt_config_str in optimizer_configs:
    # Get optimizer type (first part before underscore)
    opt_type = opt_config_str.split('_')[0]
    
    # Get final test accuracy from config_data
    test_acc_key = f'test_acc_{opt_config_str}'
    if test_acc_key in config_data:
        final_test_acc = config_data[test_acc_key]
    else:
        final_test_acc = 0.0
    
    results.append({
        'config': opt_config_str,
        'type': opt_type,
        'final_test_acc': final_test_acc
    })

print(f"\nLoaded {len(results)} optimizer results")


# Group by optimizer type
grouped = defaultdict(list)
for result in results:
    grouped[result['type']].append(result)

# Sort each group by final test accuracy and get top 3
top3_per_type = {}
for opt_type, opt_results in grouped.items():
    sorted_results = sorted(opt_results, key=lambda x: x['final_test_acc'], reverse=True)
    top3_per_type[opt_type] = sorted_results[:3]

# Display top 3 per type
print("Top 3 Optimizers by Type:\n")
for opt_type, top3 in top3_per_type.items():
    print(f"\n{'='*80}")
    print(f"Optimizer Type: {opt_type}")
    print(f"{'='*80}")
    for i, result in enumerate(top3, 1):
        print(f"\n  {i}. Test Acc: {result['final_test_acc']:.4f}")
        print(f"     Config: {result['config']}")


def load_metrics(config_str, log_dir):
    """Load metrics for a given optimizer config."""
    safe_name = config_str.replace(".", "_").replace("/", "_")
    metrics_path = os.path.join(log_dir, f"metrics_{safe_name}.json")
    
    if not os.path.exists(metrics_path):
        print(f"Warning: Metrics not found for {config_str}")
        return None
    
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    return metrics

# Load metrics for all top 3 optimizers
top3_metrics = {}
for opt_type, top3 in top3_per_type.items():
    top3_metrics[opt_type] = []
    for result in top3:
        metrics = load_metrics(result['config'], LOG_DIR)
        if metrics:
            top3_metrics[opt_type].append({
                'config': result['config'],
                'metrics': metrics,
                'final_test_acc': result['final_test_acc']
            })

print("Loaded metrics for all top performers")

In [ ]:
# Load singular value data for best optimizer of each type
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def load_singular_values(config_str, log_dir):
    """Load singular values for a given optimizer config."""
    safe_name = config_str.replace(".", "_").replace("/", "_")
    sv_path = os.path.join(log_dir, f"singular_values_{safe_name}.pkl")
    
    if not os.path.exists(sv_path):
        print(f"Warning: Singular values not found for {config_str}")
        return None
    
    with open(sv_path, 'rb') as f:
        sv_data = pickle.load(f)
    
    return sv_data

# Get best optimizer for each type
best_per_type = {}
for opt_type, top3_list in top3_metrics.items():
    # Best is the first one (already sorted by test_acc)
    best_per_type[opt_type] = top3_list[0]

# Load singular values for best optimizers
sv_data_dict = {}
for opt_type, best_config in best_per_type.items():
    sv_data = load_singular_values(best_config['config'], LOG_DIR)
    if sv_data:
        sv_data_dict[opt_type] = {
            'config': best_config['config'],
            'sv_data': sv_data,
            'test_acc': best_config['final_test_acc']
        }

print(f"Loaded singular values for {len(sv_data_dict)} optimizer types")
for opt_type, data in sv_data_dict.items():
    print(f"  {opt_type}: {len(data['sv_data'])} snapshots")

In [ ]:
# Load gradient statistics for best optimizer of each type
def load_gradient_statistics(config_str, log_dir):
    """Load gradient statistics for a given optimizer config."""
    safe_name = config_str.replace(".", "_").replace("/", "_")
    grad_stats_path = os.path.join(log_dir, f"gradient_stats_{safe_name}.pkl")
    
    if not os.path.exists(grad_stats_path):
        print(f"Warning: Gradient statistics not found for {config_str}")
        return None
    
    with open(grad_stats_path, 'rb') as f:
        grad_stats_data = pickle.load(f)
    
    return grad_stats_data

# Load gradient statistics for best optimizers (already defined in best_per_type)
grad_stats_dict = {}
for opt_type, best_config in best_per_type.items():
    grad_stats = load_gradient_statistics(best_config['config'], LOG_DIR)
    if grad_stats:
        grad_stats_dict[opt_type] = {
            'config': best_config['config'],
            'grad_stats': grad_stats,
            'test_acc': best_config['final_test_acc']
        }

if grad_stats_dict:
    print(f"Loaded gradient statistics for {len(grad_stats_dict)} optimizer types")
    for opt_type, data in grad_stats_dict.items():
        print(f"  {opt_type}: {len(data['grad_stats'])} snapshots")
        if len(data['grad_stats']) > 0:
            _, first_stats = data['grad_stats'][0]
            if 'param_name' in first_stats:
                print(f"    Tracking parameter: {first_stats['param_name']}")
else:
    print("No gradient statistics found. This feature requires CifarNet architecture.")

In [ ]:
# Load gradient spectra for all filter layers
def load_gradient_spectra(config_str, log_dir):
    '''Load gradient spectra for a given optimizer config.'''
    safe_name = config_str.replace(".", "_").replace("/", "_")
    grad_spectra_path = os.path.join(log_dir, f"gradient_spectra_{safe_name}.pkl")

    if not os.path.exists(grad_spectra_path):
        print(f"Warning: Gradient spectra not found for {config_str}")
        return None

    with open(grad_spectra_path, 'rb') as f:
        grad_spectra_data = pickle.load(f)

    return grad_spectra_data

# Load gradient spectra for best optimizers
grad_spectra_dict = {}
for opt_type, best_config in best_per_type.items():
    grad_spectra = load_gradient_spectra(best_config['config'], LOG_DIR)
    if grad_spectra:
        grad_spectra_dict[opt_type] = {
            'config': best_config['config'],
            'grad_spectra': grad_spectra,
            'test_acc': best_config['final_test_acc']
        }

if grad_spectra_dict:
    print(f"Loaded gradient spectra for {len(grad_spectra_dict)} optimizer types")

    # Print available layers
    print("\\nAvailable layers for gradient spectrum analysis:")
    print("="*80)

    for opt_type, data in grad_spectra_dict.items():
        print(f"\\n{opt_type}:")
        if len(data['grad_spectra']) > 0:
            _, first_spectra = data['grad_spectra'][1]
            layer_names = sorted(first_spectra.keys())
            print(f"  Number of snapshots: {len(data['grad_spectra'])}")
            print(f"  Number of layers tracked: {len(layer_names)}")
            print(f"  Layer names:")
            for i, layer_name in enumerate(layer_names, 1):
                print(f"    {i}. {layer_name}")
else:
    print("No gradient spectra data found.")

In [ ]:
# ========== SELECT LAYER TO ANALYZE ==========
# Change this to the layer you want to visualize
SELECTED_LAYER = "layers.1.conv1.weight"  # Will use first layer by default
# SELECTED_LAYER  = "1.weight"
# Auto-select first available layer if not specified
if grad_spectra_dict and SELECTED_LAYER is None:
    opt_type = list(grad_spectra_dict.keys())[0]
    _, first_spectra = grad_spectra_dict[opt_type]['grad_spectra'][0]
    layer_names = sorted(first_spectra.keys())
    if layer_names:
        SELECTED_LAYER = layer_names[0]
        print(f"Auto-selected layer: {SELECTED_LAYER}")
        print(f"\\nTo analyze a different layer, set SELECTED_LAYER to one of the layer names printed above")
else:
    print(f"Selected layer: {SELECTED_LAYER}")

In [ ]:
# Plot gradient spectrum evolution for selected layer (animated)
if grad_spectra_dict and SELECTED_LAYER:
    print(f"\\nCreating gradient spectrum animation for layer: {SELECTED_LAYER}")
    print("(this may take a moment)")

    num_optimizers = len(grad_spectra_dict)
    opt_types = list(grad_spectra_dict.keys())

    # Determine common time steps
    all_steps = set()
    for opt_type in opt_types:
        grad_data = grad_spectra_dict[opt_type]['grad_spectra']
        for step, spectra in grad_data:
            if SELECTED_LAYER in spectra:
                all_steps.add(step)
    common_steps = sorted(list(all_steps))

    print(f"Total frames: {len(common_steps)}")

    # Set up the figure - single plot for all optimizers
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))

    # Colors and markers for different optimizer types
    opt_colors = plt.cm.tab10(np.linspace(0, 1, num_optimizers))
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

    def update_layer_spectrum(frame_idx):
        '''Update function for layer spectrum animation.'''
        step = common_steps[frame_idx]
        ax.clear()

        # Plot all optimizers on the same axes
        for idx, opt_type in enumerate(opt_types):
            grad_data = grad_spectra_dict[opt_type]['grad_spectra']

            # Find spectrum for this step and layer
            spectrum = None
            for s, spectra in grad_data:
                if s == step and SELECTED_LAYER in spectra:
                    spectrum = spectra[SELECTED_LAYER]['spectrum']
                    break

            if spectrum is not None and len(spectrum) > 0:
                # Sort singular values in descending order
                sorted_sv = np.sort(spectrum)[::-1]
                indices = np.arange(len(sorted_sv))

                # Plot spectrum
                ax.semilogy(indices, sorted_sv,
                           label=f'{opt_type} (Acc: {grad_spectra_dict[opt_type]["test_acc"]:.4f})',
                           color=opt_colors[idx],
                           linewidth=2.5,
                           marker=markers[idx % len(markers)],
                           markersize=5,
                           markevery=max(1, len(sorted_sv)//20),
                           alpha=0.8)

        # Configure plot
        ax.set_xlabel('Singular Value Index', fontweight='bold', fontsize=14)
        ax.set_ylabel('Magnitude (log scale)', fontweight='bold', fontsize=14)
        ax.set_title(f'Gradient Spectrum: {SELECTED_LAYER}\\nStep {step}',
                    fontweight='bold', fontsize=16)
        ax.legend(loc='best', fontsize=11, framealpha=0.9)
        ax.grid(True, alpha=0.3)
        # ax.set_ylim(1e-2, 1e3)

        # Add step annotation
        ax.text(0.02, 0.02, f'Training Step: {step}', transform=ax.transAxes,
               fontsize=12, verticalalignment='bottom', fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

        return []

    # Create animation
    print("Rendering layer spectrum animation frames...")
    anim_layer_spectrum = FuncAnimation(fig, update_layer_spectrum, frames=len(common_steps),
                                        interval=200, blit=False, repeat=True)

    # Save as HTML5 video for notebook display
    plt.tight_layout()
    html_anim_layer = HTML(anim_layer_spectrum.to_jshtml())

    # Also save as GIF
    layer_safe_name = SELECTED_LAYER.replace(".", "_").replace("/", "_")
    gif_path_layer = os.path.join(LOG_DIR, f'gradient_spectrum_{layer_safe_name}_evolution.gif')
    print(f"Saving layer spectrum animation to {gif_path_layer}...")
    anim_layer_spectrum.save(gif_path_layer, writer='pillow', fps=5, dpi=100)
    print(f"Layer spectrum animation saved!")

    # Display in notebook
    display(html_anim_layer)
else:
    print("Skipping layer gradient spectrum animation - no data available or no layer selected")


In [ ]:
# Plot gradient norms evolution for selected layer (static plots)
if grad_spectra_dict and SELECTED_LAYER:
    num_optimizers = len(grad_spectra_dict)
    opt_types = list(grad_spectra_dict.keys())

    # Create figure with 3 rows (spectral norm, frobenius norm, trace)
    fig, axes = plt.subplots(3, 1, figsize=(16, 14))

    # Colors for different optimizer types
    opt_colors = plt.cm.tab10(np.linspace(0, 1, num_optimizers))

    # Extract data for plotting
    for idx, opt_type in enumerate(opt_types):
        grad_data = grad_spectra_dict[opt_type]['grad_spectra']

        # Extract steps and statistics for selected layer
        steps = []
        spectral_norms = []
        frobenius_norms = []
        traces = []

        for step, spectra in grad_data:
            if SELECTED_LAYER in spectra:
                layer_stats = spectra[SELECTED_LAYER]
                steps.append(step)
                spectral_norms.append(layer_stats.get('spectral_norm', np.nan))
                frobenius_norms.append(layer_stats.get('frobenius_norm', np.nan))
                traces.append(np.sum(layer_stats.get('spectrum', np.nan)))

        # Convert to numpy arrays
        steps = np.array(steps)
        spectral_norms = np.array(spectral_norms)
        frobenius_norms = np.array(frobenius_norms)
        traces = np.array(traces)

        # Plot spectral norm
        axes[0].plot(steps, spectral_norms,
                     label=f'{opt_type} (Acc: {grad_spectra_dict[opt_type]["test_acc"]:.4f})',
                     color=opt_colors[idx], linewidth=2.5, marker='o', markersize=6,
                     markevery=max(1, len(steps)//15))

        # Plot Frobenius norm
        axes[1].plot(steps, frobenius_norms,
                     label=f'{opt_type} (Acc: {grad_spectra_dict[opt_type]["test_acc"]:.4f})',
                     color=opt_colors[idx], linewidth=2.5, marker='s', markersize=6,
                     markevery=max(1, len(steps)//15))

        # Plot trace
        axes[2].plot(steps, traces,
                     label=f'{opt_type} (Acc: {grad_spectra_dict[opt_type]["test_acc"]:.4f})',
                     color=opt_colors[idx], linewidth=2.5, marker='^', markersize=6,
                     markevery=max(1, len(steps)//15))

    # Configure spectral norm plot
    axes[0].set_xlabel('Training Step', fontweight='bold', fontsize=13)
    axes[0].set_ylabel('Spectral Norm ||∇||₂', fontweight='bold', fontsize=13)
    axes[0].set_title(f'Gradient Spectral Norm: {SELECTED_LAYER}',
                      fontweight='bold', fontsize=15)
    axes[0].legend(loc='best', fontsize=11, framealpha=0.9)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_yscale('log')

    # Configure Frobenius norm plot
    axes[1].set_xlabel('Training Step', fontweight='bold', fontsize=13)
    axes[1].set_ylabel('Frobenius Norm ||∇||_F', fontweight='bold', fontsize=13)
    axes[1].set_title(f'Gradient Frobenius Norm: {SELECTED_LAYER}',
                      fontweight='bold', fontsize=15)
    axes[1].legend(loc='best', fontsize=11, framealpha=0.9)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_yscale('log')

    # Configure trace plot
    axes[2].set_xlabel('Training Step', fontweight='bold', fontsize=13)
    axes[2].set_ylabel('Trace Tr(∇ᵀ∇)', fontweight='bold', fontsize=13)
    axes[2].set_title(f'Gradient Gram Matrix Trace: {SELECTED_LAYER}',
                      fontweight='bold', fontsize=15)
    axes[2].legend(loc='best', fontsize=11, framealpha=0.9)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_yscale('log')

    plt.tight_layout()

    layer_safe_name = SELECTED_LAYER.replace(".", "_").replace("/", "_")
    save_path = os.path.join(LOG_DIR, f'gradient_norms_{layer_safe_name}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\\nGradient norms plot saved to: {save_path}")
else:
    print("Skipping layer gradient norms plots - no data available or no layer selected")


In [ ]:
# Publication-quality figure: Training dynamics and gradient properties
if grad_spectra_dict and SELECTED_LAYER and best_per_type:
    print("Creating publication-quality training dynamics figure...")
    
    # Set up publication style
    plt.rcParams.update({
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 13,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 9,
        'figure.titlesize': 14,
        'lines.linewidth': 2.0,
        'lines.markersize': 4,
    })
    
    # Create 2x2 subplot
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    # Get optimizer types and colors
    opt_types = list(grad_spectra_dict.keys())
    num_optimizers = len(opt_types)
    opt_colors = plt.cm.tab10(np.linspace(0, 1, num_optimizers))
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']
    
    # Plot for each optimizer
    for idx, opt_type in enumerate(opt_types):
        color = opt_colors[idx]
        marker = markers[idx % len(markers)]
        label = f'{opt_type} (Acc: {grad_spectra_dict[opt_type]["test_acc"]:.2f}%)'
        
        # === 1. Training Loss ===
        if opt_type in best_per_type and 'metrics' in best_per_type[opt_type]:
            metrics = best_per_type[opt_type]['metrics']
            if 'train_loss' in metrics and len(metrics['train_loss']) > 0:
                # Unpack [step, loss] pairs
                train_data = np.array(metrics['train_loss'])
                if train_data.ndim == 2 and train_data.shape[1] == 2:
                    steps = train_data[:, 0]
                    losses = train_data[:, 1]
                else:
                    steps = np.arange(len(train_data))
                    losses = train_data
                
                axes[0].plot(steps, losses, color=color, marker=marker,
                           markevery=max(1, len(steps)//15), label=label, alpha=0.85)
        
        # === 2. Validation Accuracy ===
        if opt_type in best_per_type and 'metrics' in best_per_type[opt_type]:
            metrics = best_per_type[opt_type]['metrics']
            if 'val_acc' in metrics and len(metrics['val_acc']) > 0:
                # Unpack [step, acc] pairs
                val_data = np.array(metrics['val_acc'])
                if val_data.ndim == 2 and val_data.shape[1] == 2:
                    steps = val_data[:, 0]
                    accs = val_data[:, 1]
                else:
                    steps = np.arange(len(val_data))
                    accs = val_data
                
                axes[1].plot(steps, accs, color=color, marker=marker,
                           markevery=max(1, len(steps)//15), label=label, alpha=0.85)
        
        # === 3. Nuclear Norm (trace of singular values) ===
        if opt_type in grad_spectra_dict:
            grad_data = grad_spectra_dict[opt_type]['grad_spectra']
            steps = []
            nuclear_norms = []
            
            for step, spectra in grad_data:
                if SELECTED_LAYER in spectra:
                    layer_stats = spectra[SELECTED_LAYER]
                    spectrum = layer_stats.get('spectrum', [])
                    if len(spectrum) > 0:
                        steps.append(step)
                        # Nuclear norm = sum of singular values (trace)
                        nuclear_norms.append(np.sum(spectrum))
            
            if len(steps) > 0:
                axes[2].plot(steps, nuclear_norms, color=color, marker=marker,
                           markevery=max(1, len(steps)//15), label=label, alpha=0.85)
        
        # === 4. Top Singular Value (spectral norm) ===
        if opt_type in grad_spectra_dict:
            grad_data = grad_spectra_dict[opt_type]['grad_spectra']
            steps = []
            top_sv = []
            
            for step, spectra in grad_data:
                if SELECTED_LAYER in spectra:
                    layer_stats = spectra[SELECTED_LAYER]
                    spectral_norm = layer_stats.get('spectral_norm', None)
                    if spectral_norm is not None:
                        steps.append(step)
                        top_sv.append(spectral_norm)
            
            if len(steps) > 0:
                axes[3].plot(steps, top_sv, color=color, marker=marker,
                           markevery=max(1, len(steps)//15), label=label, alpha=0.85)
    
    # === Configure subplots ===
    
    # Training Loss
    axes[0].set_xlabel('Training Step', fontweight='bold')
    axes[0].set_ylabel('Training Loss', fontweight='bold')
    axes[0].set_title('(a) Training Loss', fontweight='bold', loc='left')
    axes[0].legend(loc='best', framealpha=0.95)
    axes[0].grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    axes[0].set_yscale('log')
    
    # Validation Accuracy
    axes[1].set_xlabel('Training Step', fontweight='bold')
    axes[1].set_ylabel('Validation Accuracy (%)', fontweight='bold')
    axes[1].set_title('(b) Validation Accuracy', fontweight='bold', loc='left')
    axes[1].legend(loc='best', framealpha=0.95)
    axes[1].grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Nuclear Norm
    axes[2].set_xlabel('Training Step', fontweight='bold')
    axes[2].set_ylabel('Nuclear Norm ||∇||₊', fontweight='bold')
    axes[2].set_title(f'(c) Nuclear Norm: {SELECTED_LAYER}', fontweight='bold', loc='left')
    axes[2].legend(loc='best', framealpha=0.95)
    axes[2].grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    axes[2].set_yscale('log')
    
    # Top Singular Value
    axes[3].set_xlabel('Training Step', fontweight='bold')
    axes[3].set_ylabel('Top Singular Value σ₁', fontweight='bold')
    axes[3].set_title(f'(d) Top Singular Value: {SELECTED_LAYER}', fontweight='bold', loc='left')
    axes[3].legend(loc='best', framealpha=0.95)
    axes[3].grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    axes[3].set_yscale('log')
    
    # Overall title
    fig.suptitle('Training Dynamics and Gradient Properties Across Optimizers',
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Tight layout with spacing
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    
    # Save figure
    layer_safe_name = SELECTED_LAYER.replace(".", "_").replace("/", "_")
    save_path = os.path.join(LOG_DIR, f'training_dynamics_{layer_safe_name}_publication.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    # Also save as PDF for paper submission
    pdf_path = os.path.join(LOG_DIR, f'training_dynamics_{layer_safe_name}_publication.pdf')
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    plt.show()
    
    print(f"\n✓ Publication figure saved:")
    print(f"  PNG: {save_path}")
    print(f"  PDF: {pdf_path}")
else:
    print("Skipping publication figure - missing required data")